<a href="https://colab.research.google.com/github/fjessenia/mobilejknpaperver2/blob/main/mobile_jkn_mixed_sentiment_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile JKN — Mixed-Sentiment Benchmark
**Positive / Negative / Mixed** · TF-IDF classifiers vs transformer (IndoBERT, IndoBERTweet) vs LLM · evaluasi berulang, group-aware, tanpa leakage

In [2]:
import os, sys, re, json, random, warnings, subprocess, time, math
SMOKE = os.environ.get('JKN_SMOKE') == '1'
IN_COLAB = 'google.colab' in sys.modules
def _pip(*pkgs): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
if IN_COLAB:
    _pip('Sastrawi', 'imbalanced-learn', 'transformers', 'accelerate', 'anthropic')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg') if SMOKE else None
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, cohen_kappa_score, make_scorer)

# ====================== CONFIG ======================
CFG = dict(
    seed=42,
    kaggle_csv='kaggle_Mobile-JKN_Review.csv',        # file Kaggle asli (5.000 baris)
    annot_csv='dataset_annotated_final.csv',          # hasil anotasi (upload setelah anotasi selesai)
    out_dir='outputs',
    n_a2=500,                                         # jumlah review yang dilabeli Annotator 2
    drop_auto_neutral=False,                          # False: review berlabel otomatis neutral TETAP dianotasi (tidak ada filter berbasis label otomatis)
    overwrite_template=False,                         # True: timpa data_for_annotation.csv yang sudah ada (HATI-HATI)
    n_splits=5, n_repeats=3, inner_splits=3,
    ros_ratio=1.0,                                    # oversampling: 1.0 = seimbang penuh
    use_smote=True,                                   # butuh imbalanced-learn
    stopword_variant='keep_negation_contrast',        # varian utama (ablasi menjalankan ketiganya)
    run_bert=True, bert_repeats=1, bert_epochs=4, bert_lr=2e-5, bert_bs=16, bert_val_frac=0.1,
    bert_models={'IndoBERT': 'indobenchmark/indobert-base-p1',
                 'IndoBERTweet': 'indolem/indobertweet-base-uncased'},
    run_llm=False, llm_model='claude-haiku-5-5', llm_modes=['zero_shot', 'few_shot'], llm_shots_per_class=2,
)
if SMOKE:   # hanya untuk uji kode, bukan untuk hasil paper
    CFG.update(kaggle_csv=os.environ['JKN_KAGGLE'], annot_csv=os.environ['JKN_ANNOT'], out_dir=os.environ['JKN_OUT'],
               n_a2=120, n_splits=3, n_repeats=1, inner_splits=2, run_bert=False, use_smote=False)
os.makedirs(CFG['out_dir'], exist_ok=True)
SEED = CFG['seed']; random.seed(SEED); np.random.seed(SEED)
LABELS = ['positive', 'negative', 'mixed']
print('Config:', json.dumps(CFG, indent=1))

Config: {
 "seed": 42,
 "kaggle_csv": "kaggle_Mobile-JKN_Review.csv",
 "annot_csv": "dataset_annotated_final.csv",
 "out_dir": "outputs",
 "n_a2": 500,
 "drop_auto_neutral": false,
 "overwrite_template": false,
 "n_splits": 5,
 "n_repeats": 3,
 "inner_splits": 3,
 "ros_ratio": 1.0,
 "use_smote": true,
 "stopword_variant": "keep_negation_contrast",
 "run_bert": true,
 "bert_repeats": 1,
 "bert_epochs": 4,
 "bert_lr": 2e-05,
 "bert_bs": 16,
 "bert_val_frac": 0.1,
 "bert_models": {
  "IndoBERT": "indobenchmark/indobert-base-p1",
  "IndoBERTweet": "indolem/indobertweet-base-uncased"
 },
 "run_llm": false,
 "llm_model": "claude-haiku-5-5",
 "llm_modes": [
  "zero_shot",
  "few_shot"
 ],
 "llm_shots_per_class": 2
}


## 1. Preprocessing

In [3]:
SLANG = {
    'gk': 'tidak', 'gak': 'tidak', 'ga': 'tidak', 'tdk': 'tidak', 'nggak': 'tidak', 'g': 'tidak', 'gx': 'tidak',
    'ngak': 'tidak', 'gabisa': 'tidak bisa', 'gakbisa': 'tidak bisa', 'jgn': 'jangan', 'bkn': 'bukan',
    'yg': 'yang', 'tp': 'tapi', 'tpi': 'tapi', 'lg': 'lagi', 'lgi': 'lagi', 'jd': 'jadi', 'jdi': 'jadi', 'sy': 'saya',
    'sya': 'saya', 'dgn': 'dengan', 'org': 'orang', 'gt': 'gitu', 'gth': 'gitu', 'dr': 'dari', 'sm': 'sama', 'sma': 'sama',
    'krn': 'karena', 'karna': 'karena', 'krna': 'karena', 'utk': 'untuk', 'bwt': 'buat', 'gmn': 'gimana', 'knp': 'kenapa',
    'klo': 'kalau', 'klau': 'kalau', 'trus': 'terus', 'tros': 'terus', 'dpt': 'dapat', 'mgkn': 'mungkin', 'kpn': 'kapan',
    'bgus': 'bagus', 'bgt': 'banget', 'bgtu': 'begitu', 'dftr': 'daftar', 'antri': 'antrean', 'antren': 'antrean',
    'antrian': 'antrean', 'lemot': 'lambat', 'eror': 'error', 'erorr': 'error', 'errorr': 'error', 'pake': 'pakai',
    'pakek': 'pakai', 'pakae': 'pakai', 'mulu': 'selalu', 'blm': 'belum', 'sdh': 'sudah', 'udh': 'sudah', 'udah': 'sudah',
    'aplikasiny': 'aplikasinya', 'app': 'aplikasi', 'apk': 'aplikasi', 'stres': 'stres', 'stress': 'stres',
    'blng': 'bilang', 'plis': 'tolong', 'pliss': 'tolong'}

def clean_and_normalize(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return ' '.join(SLANG.get(w, w) for w in text.split())

NEGATION = {'tidak', 'bukan', 'jangan', 'kurang', 'belum', 'tanpa', 'nggak'}
CONTRAST = {'tapi', 'tetapi', 'namun', 'meski', 'meskipun', 'padahal', 'walau', 'walaupun', 'sayang', 'sayangnya',
            'cuma', 'kadang', 'kecuali', 'sedangkan'}
DOMAIN_STOP = {'aplikasi', 'jkn', 'mobile', 'bpjs', 'nya'}
FALLBACK_STOP = set('di dan ke yang untuk dari ini itu dengan atau adalah pada juga ada saya aku kami kita anda sebuah para oleh dalam akan agar saat ya yah lagi sama bisa mau karena jadi sudah masih lebih banyak'.split())
try:
    import nltk
    try: nltk.data.find('corpora/stopwords')
    except LookupError: nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    NLTK_STOP = set(stopwords.words('indonesian'))
except Exception as e:
    print('NLTK stopwords tidak tersedia, memakai daftar fallback:', e); NLTK_STOP = FALLBACK_STOP
print('Kata kontras yang ADA di daftar stopword NLTK (dibuang oleh daftar NLTK default):', sorted(NLTK_STOP & CONTRAST))
print('Kata negasi yang ada di daftar stopword NLTK:', sorted(NLTK_STOP & NEGATION))

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _stemmer = StemmerFactory().create_stemmer(); STEM_OK = True
except Exception as e:
    print('Sastrawi tidak tersedia -> TANPA stemming (hanya untuk uji kode):', e); _stemmer = None; STEM_OK = False
_stem_cache = {}
def stem_token(w):
    if _stemmer is None: return w
    if w not in _stem_cache: _stem_cache[w] = _stemmer.stem(w)
    return _stem_cache[w]

def stop_set(variant):
    if variant == 'nltk_default':              return (NLTK_STOP - NEGATION) | DOMAIN_STOP
    if variant == 'keep_negation_contrast': return (NLTK_STOP - NEGATION - CONTRAST) | DOMAIN_STOP
    if variant == 'no_stopwords':           return set()
    raise ValueError(variant)

def preprocess_ml(text_clean, variant):
    sw = stop_set(variant); minlen = 2 if variant != 'nltk_default' else 3   # varian 'nltk_default' membuang token <=2 huruf
    toks = [w for w in text_clean.split() if w not in sw and len(w) >= minlen]
    return ' '.join(stem_token(w) for w in toks)

CONTRAST_CORE = {'tapi', 'tetapi', 'namun'}      # 'tp'/'tpi' sudah dinormalisasi menjadi 'tapi' oleh kamus slang
def has_contrast(text_clean): return bool(set(text_clean.split()) & CONTRAST)
def has_contrast_core(text_clean): return bool(set(text_clean.split()) & CONTRAST_CORE)
def n_words(t): return len(t.split())

Kata kontras yang ADA di daftar stopword NLTK (dibuang oleh daftar NLTK default): ['cuma', 'meski', 'meskipun', 'namun', 'padahal', 'sedangkan', 'tapi', 'tetapi', 'walau', 'walaupun']
Kata negasi yang ada di daftar stopword NLTK: ['belum', 'bukan', 'jangan', 'kurang', 'tanpa', 'tidak']


## 2. Bagian A — Siapkan file anotasi
**Definisi tugas & label** (cantumkan di paper): annotator melabeli dari nol dengan lima kategori, berdasarkan kolom `content` (teks asli, termasuk emoji), bukan `content_clean`.
- `positive` — hanya pujian/puas · `negative` — hanya keluhan/kritik · `mixed` — memuat penilaian positif **dan** negatif sekaligus (mis. "bagus tapi sering error")
- `neutral` — tidak ada penilaian (pertanyaan, informasi)

`neutral` dikeluarkan dari benchmark karena pertanyaan penelitiannya tiga kelas (positive/negative/mixed). Konsekuensinya tugas menjadi lebih "bersih" daripada kondisi nyata — tulis di limitations.
**Annotator tidak boleh melihat** rating bintang, label otomatis Kaggle, atau rationale CoT; semuanya sengaja dipisah ke `data_meta.csv`.
Filter sebelum anotasi hanya membuang review ≤3 karakter setelah cleaning. Review berlabel otomatis *neutral* **tetap dianotasi** (kecuali `drop_auto_neutral=True`), karena rating 3 adalah tempat yang masuk akal untuk review mixed.

In [4]:
def read_any(path):
    if str(path).lower().endswith(('.xlsx', '.xls')): return pd.read_excel(path)
    return pd.read_csv(path, encoding='utf-8-sig')

def build_candidates(raw):
    """Review yang masuk anotasi. Deterministik: urutan diacak dengan seed tetap, row_id = nomor baris file Kaggle."""
    short = raw['content_clean'].str.len() <= 3
    auto = raw['sentiment_polarity'].astype(str).str.lower() if 'sentiment_polarity' in raw.columns else pd.Series('', index=raw.index)
    drop = short | (auto.eq('neutral') if CFG['drop_auto_neutral'] else False)
    return raw[~drop].sample(frac=1, random_state=SEED).reset_index(drop=True), short, auto

raw = read_any(CFG['kaggle_csv'])
raw['content_clean'] = raw['content'].map(clean_and_normalize)
raw['row_id'] = np.arange(1, len(raw) + 1)
cand, short, auto = build_candidates(raw)

auto_neu_kept = int((auto.eq('neutral') & ~short).sum())
rows = [('Review mentah (Kaggle)', len(raw)), ('− terlalu pendek setelah cleaning (≤3 karakter)', -int(short.sum()))]
if CFG['drop_auto_neutral']: rows.append(('− label otomatis Kaggle = neutral (dibuang SEBELUM anotasi)', -auto_neu_kept))
rows.append(('= masuk anotasi', len(cand)))
if not CFG['drop_auto_neutral']: rows.append(('   (di antaranya berlabel otomatis neutral)', auto_neu_kept))
funnel_A = rows
print(pd.DataFrame(funnel_A, columns=['langkah', 'n']).to_string(index=False))

# rentang waktu data mentah
DATE_COL = next((c for c in ['at', 'date', 'created_at', 'review_date', 'timestamp', 'tanggal'] if c in raw.columns), None)
if DATE_COL:
    dts = pd.to_datetime(raw[DATE_COL], errors='coerce')
    print(f"\nRentang waktu data mentah ({DATE_COL}): {dts.min():%d %b %Y} – {dts.max():%d %b %Y} ({(dts.max() - dts.min()).days + 1} hari)")
else: print('\nKolom tanggal tidak ditemukan -> isi rentang waktu secara manual di paper.')

# ---- file untuk annotator: HANYA teks (tanpa rating / label otomatis / rationale)
tpl = cand[['row_id', 'content', 'content_clean']].copy()
rng = np.random.default_rng(SEED)
a2_ids = set(rng.choice(tpl['row_id'].values, min(CFG['n_a2'], len(tpl)), replace=False).tolist())
tpl['annotator2_subset'] = tpl['row_id'].isin(a2_ids).astype(int)
for c in ['label_annotator1', 'label_annotator2', 'catatan']: tpl[c] = ''
a2_file = tpl[tpl['annotator2_subset'] == 1][['row_id', 'content', 'content_clean', 'label_annotator2', 'catatan']]
meta = cand.drop(columns=['content', 'content_clean'])           # rating, label otomatis, tanggal, rationale, dll.

def safe_write(df, name, always=False):
    p = f"{CFG['out_dir']}/{name}"
    if os.path.exists(p) and not always and not CFG['overwrite_template']:
        print(f'  {name}: sudah ada -> TIDAK ditimpa'); returnaa
    df.to_csv(p, index=False, encoding='utf-8-sig'); print(f'  {name}: ditulis ({len(df)} baris)')
print('\nFile dihasilkan di', CFG['out_dir'])
safe_write(tpl, 'data_for_annotation.csv')
safe_write(a2_file, 'data_for_annotator2.csv')
safe_write(meta, 'data_meta.csv', always=True)
pd.DataFrame(funnel_A, columns=['langkah', 'n']).to_csv(f"{CFG['out_dir']}/data_funnel_before_annotation.csv", index=False)
print(f"\nBerikan ke Annotator 1: data_for_annotation.csv ({len(tpl)} baris, isi kolom label_annotator1)")
print(f"Berikan ke Annotator 2: data_for_annotator2.csv ({len(a2_file)} baris, isi kolom label_annotator2)")
print("JANGAN berikan data_meta.csv ke annotator. Label yang sah: positive / negative / mixed / neutral.")

                                        langkah    n
                         Review mentah (Kaggle) 5000
− terlalu pendek setelah cleaning (≤3 karakter) -198
                                = masuk anotasi 4802
       (di antaranya berlabel otomatis neutral)   92

Rentang waktu data mentah (at): 09 Sep 2026 – 17 Oct 2026 (39 hari)

File dihasilkan di outputs
  data_for_annotation.csv: ditulis (4802 baris)
  data_for_annotator2.csv: ditulis (500 baris)
  data_meta.csv: ditulis (4802 baris)

Berikan ke Annotator 1: data_for_annotation.csv (4802 baris, isi kolom label_annotator1)
Berikan ke Annotator 2: data_for_annotator2.csv (500 baris, isi kolom label_annotator2)
JANGAN berikan data_meta.csv ke annotator. Label yang sah: positive / negative / mixed / neutral.


**(Opsional) Menggabungkan hasil dua annotator** menjadi `dataset_annotated_final.csv` (jalankan manual setelah kedua file terisi; ganti nama file):

In [5]:
def merge_annotations(a1_path, a2_path, out_path=CFG['annot_csv']):
    t = read_any(a1_path).drop(columns=['label_annotator2'], errors='ignore')
    a2 = read_any(a2_path)[['row_id', 'label_annotator2']]
    t.merge(a2, on='row_id', how='left', validate='1:1').to_csv(out_path, index=False, encoding='utf-8-sig')

merge_annotations('outputs/data_for_annotation_filled.csv',
                  'outputs/data_for_annotator2_filled.csv')

## 3. Bagian B — Muat hasil anotasi, funnel akhir, kappa, provenance
Mulai dari sini notebook membutuhkan `dataset_annotated_final.csv`.

In [6]:
if not os.path.exists(CFG['annot_csv']):
    raise RuntimeError(f"'{CFG['annot_csv']}' belum ada. Selesaikan anotasi (lihat §2), upload file tersebut, lalu jalankan ulang dengan Runtime -> Run all.")
ann = read_any(CFG['annot_csv'])
assert {'row_id', 'content', 'label_annotator1'} <= set(ann.columns), 'Kolom wajib: row_id, content, label_annotator1 (label_annotator2 untuk kappa)'
LABEL_MAP = {'positif': 'positive', 'negatif': 'negative', 'campuran': 'mixed', 'netral': 'neutral'}
VALID = set(LABELS) | {'neutral'}
for c in ['label_annotator1', 'label_annotator2']:
    if c not in ann.columns: ann[c] = 'nan'
    ann[c] = ann[c].fillna('nan').astype(str).str.lower().str.strip().replace({'': 'nan'}).replace(LABEL_MAP)
    bad = sorted(set(ann[c]) - VALID - {'nan'})
    if bad: print(f'PERINGATAN: label tidak dikenal di {c}: {bad}')

# ---- gabungkan dengan metadata (rating, label otomatis, tanggal) HANYA setelah anotasi selesai
ref = build_candidates(raw)[0].set_index('row_id')
unk = ~ann['row_id'].isin(ref.index)
mism = int((ann.loc[~unk, 'row_id'].map(ref['content']).astype(str).values != ann.loc[~unk, 'content'].astype(str).values).sum())
print(f"row_id tidak ditemukan di data Kaggle: {int(unk.sum())} | isi 'content' berbeda dari sumber: {mism} (selisih kecil bisa akibat spasi/encoding Excel)")
meta_cols = [c for c in ref.columns if c not in ('content', 'content_clean') and c not in ann.columns]
ann = ann.merge(ref[meta_cols].reset_index(), on='row_id', how='left', validate='m:1')
ann['content_clean'] = ann['content'].map(clean_and_normalize)

# ---- funnel lengkap (angka eksak untuk bagian Data di paper)
a1 = ann['label_annotator1']
n_other = int((~a1.isin(VALID)).sum())
funnel = funnel_A + [('Baris pada file anotasi', len(ann)),
                     ('− Annotator 1 = neutral', -int((a1 == 'neutral').sum()))]
if n_other: funnel.append(('− Annotator 1 kosong / label tidak dikenal', -n_other))
funnel.append(('= data final (positive / negative / mixed)', int(a1.isin(LABELS).sum())))
FUNNEL = pd.DataFrame(funnel, columns=['langkah', 'n']); print(FUNNEL.to_string(index=False))
FUNNEL.to_csv(f"{CFG['out_dir']}/data_funnel.csv", index=False)
print('\nDistribusi label annotator 1 (semua kategori):\n', a1.value_counts().to_string())
data = ann[a1.isin(LABELS)].reset_index(drop=True)
print(f"\nData final: {len(data)} review ->", data['label_annotator1'].value_counts().to_dict())

# ---- Kappa: n subset, kategori, semua kategori, 3 kelas, one-vs-rest tiap kelas + CI bootstrap
def boot_ci(a, b, B=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(a); bs = []
    for _ in range(B):
        ix = rng.integers(0, n, n)
        if len(set(a[ix]) | set(b[ix])) > 1: bs.append(cohen_kappa_score(a[ix], b[ix]))
    return [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))] if bs else [float('nan'), float('nan')]
def kappa_block(ann):
    d = ann[~ann['label_annotator2'].isin(['nan']) & ~ann['label_annotator1'].isin(['nan'])].reset_index(drop=True)
    if len(d) == 0: print('Tidak ada baris yang dilabeli kedua annotator -> kappa dilewati'); return None
    a1, a2 = d['label_annotator1'].values, d['label_annotator2'].values
    out = dict(n_double_annotated=len(d), pct_of_all_annotated=round(100 * len(d) / len(ann), 1), categories=sorted(set(a1) | set(a2)),
               pct_agreement=float((a1 == a2).mean()), kappa_all_categories=cohen_kappa_score(a1, a2), kappa_all_categories_CI95=boot_ci(a1, a2),
               label_counts_A1=pd.Series(a1).value_counts().to_dict(), label_counts_A2=pd.Series(a2).value_counts().to_dict())
    t = d[d['label_annotator1'].isin(LABELS) & d['label_annotator2'].isin(LABELS)]
    out['n_target_only'] = len(t); out['kappa_3class_target_only'] = cohen_kappa_score(t['label_annotator1'], t['label_annotator2'])
    for cls in LABELS:     # one-vs-rest: kappa untuk SETIAP kelas, agar mixed tidak "tertutup" pos/neg
        m1, m2 = (a1 == cls).astype(int), (a2 == cls).astype(int)
        k = cohen_kappa_score(m1, m2) if len(set(m1) | set(m2)) > 1 else float('nan')
        out[f'kappa_{cls}_vs_rest'] = dict(kappa=k, CI95=boot_ci(m1, m2), n_A1=int(m1.sum()), n_A2=int(m2.sum()), n_both=int((m1 & m2).sum()))
    print('Tabel silang A1 (baris) x A2 (kolom), subset ganda:\n', pd.crosstab(pd.Series(a1, name='A1'), pd.Series(a2, name='A2')).to_string())
    nm = out['kappa_mixed_vs_rest']['n_A1']
    if nm < 20: print(f'\nPERINGATAN: hanya {nm} review mixed (A1) di subset ganda -> CI kappa mixed akan sangat lebar. '
                      'Pertimbangkan menambah subset Annotator 2 dengan semua review mixed milik Annotator 1.')
    return out
kap = kappa_block(ann); print('\nKappa:', json.dumps(kap, indent=1, default=str))
json.dump(kap, open(f"{CFG['out_dir']}/kappa.json", 'w'), indent=1, default=str)

# ---- Provenance: label otomatis & rating DIPISAH dari annotator (blind); bagian ini hanya statistik deskriptif
if 'sentiment_polarity' in ann.columns:
    both = ann[a1.isin(VALID)]; auto_l = both['sentiment_polarity'].astype(str).str.lower()
    print('\nKesesuaian Annotator 1 dengan label otomatis Kaggle (annotator TIDAK melihatnya): %.1f%%' % (100 * (auto_l == both['label_annotator1']).mean()))
    print(pd.crosstab(auto_l, both['label_annotator1'], margins=True).to_string())
    an = both[auto_l == 'neutral']
    if len(an): print(f"\nDi antara {len(an)} review berlabel otomatis neutral: mixed menurut A1 = {(an.label_annotator1 == 'mixed').sum()} ({100 * (an.label_annotator1 == 'mixed').mean():.1f}%) "
                      f"-> inilah mixed yang akan hilang jika filter label otomatis dipakai sebelum anotasi")
    if 'score' in ann.columns: print('\nRating bintang x label Annotator 1:\n', pd.crosstab(both['score'], both['label_annotator1']).to_string())

row_id tidak ditemukan di data Kaggle: 0 | isi 'content' berbeda dari sumber: 1 (selisih kecil bisa akibat spasi/encoding Excel)
                                        langkah    n
                         Review mentah (Kaggle) 5000
− terlalu pendek setelah cleaning (≤3 karakter) -198
                                = masuk anotasi 4802
       (di antaranya berlabel otomatis neutral)   92
                        Baris pada file anotasi 4802
                        − Annotator 1 = neutral  -28
     = data final (positive / negative / mixed) 4774

Distribusi label annotator 1 (semua kategori):
 label_annotator1
positive    2387
negative    2302
mixed         85
neutral       28

Data final: 4774 review -> {'positive': 2387, 'negative': 2302, 'mixed': 85}
Tabel silang A1 (baris) x A2 (kolom), subset ganda:
 A2        mixed  negative  neutral  positive
A1                                          
mixed         8         0        0         2
negative      0       242        0         3
ne

## 4. Fitur, group id, split berulang

In [7]:
data['text_bert'] = data['content_clean']                       # IndoBERT: TANPA stemming/stopword removal
for v in ['nltk_default', 'keep_negation_contrast', 'no_stopwords']:
    data[f'text_ml_{v}'] = data['content_clean'].map(lambda t, v=v: preprocess_ml(t, v))
data['n_words'] = data['content'].astype(str).map(n_words)
data['has_contrast'] = data['content_clean'].map(has_contrast)
data['has_contrast_core'] = data['content_clean'].map(has_contrast_core)
assert (data['text_bert'] == data['content_clean']).all()     # transformer: TANPA stemming Sastrawi / stopword removal
y = data['label_annotator1'].values
groups = pd.factorize(data['content_clean'].str.strip())[0]       # duplikat => grup yang sama => tidak terpisah train/test
print(f'{len(data)} review, {len(set(groups))} grup unik (duplikat: {len(data) - len(set(groups))} baris)')
print('Kata kontras muncul pada %.0f%% review mixed vs %.0f%% pos/neg' % (
    100 * data.loc[y == 'mixed', 'has_contrast'].mean(), 100 * data.loc[y != 'mixed', 'has_contrast'].mean()))
print('Median panjang (kata):', data.groupby('label_annotator1')['n_words'].median().to_dict())
_sh = data['n_words'] <= 3
print(f"Review <=3 kata: {100 * _sh.mean():.1f}% data | komposisi kelas di dalamnya: {data.loc[_sh, 'label_annotator1'].value_counts(normalize=True).round(3).to_dict()}")
print('Panjang minimum review mixed:', int(data.loc[data.label_annotator1 == 'mixed', 'n_words'].min()), 'kata | baseline mayoritas (akurasi): %.3f' % data['label_annotator1'].value_counts(normalize=True).max())

N_REP = CFG['n_repeats']
SPLITS = [list(StratifiedGroupKFold(CFG['n_splits'], shuffle=True, random_state=SEED + r).split(np.zeros(len(y)), y, groups)) for r in range(N_REP)]
print('Mixed per test-fold (repeat 0):', [int((y[te] == 'mixed').sum()) for _, te in SPLITS[0]])

def prf(yt, yp):
    P, R, F, _ = precision_recall_fscore_support(yt, yp, labels=LABELS, zero_division=0)
    return dict(acc=accuracy_score(yt, yp), macro_f1=F.mean(), weighted_f1=f1_score(yt, yp, average='weighted', zero_division=0),
                F_pos=F[0], F_neg=F[1], P_mixed=P[2], R_mixed=R[2], F_mixed=F[2])
def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'), float('nan'))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)

OOF = {}      # key -> list (per repeat) of prediction arrays aligned with data rows
def store(key, rep, idx, pred):
    OOF.setdefault(key, {}).setdefault(rep, np.empty(len(y), dtype=object))[idx] = pred

4774 review, 3728 grup unik (duplikat: 1046 baris)
Kata kontras muncul pada 88% review mixed vs 11% pos/neg
Median panjang (kata): {'mixed': 18.0, 'negative': 13.0, 'positive': 2.0}
Review <=3 kata: 37.1% data | komposisi kelas di dalamnya: {'positive': 0.861, 'negative': 0.139}
Panjang minimum review mixed: 4 kata | baseline mayoritas (akurasi): 0.500
Mixed per test-fold (repeat 0): [19, 12, 17, 18, 19]


## 5. Baseline sederhana (di-fit hanya pada train fold bila perlu)

In [8]:
def run_baselines():
    for rep, folds in enumerate(SPLITS):
        for tr, te in folds:
            maj = pd.Series(y[tr]).value_counts().idxmax(); store('baseline: majority', rep, te, [maj] * len(te))
            # rating bintang -> label mayoritas per bintang (dipelajari di train fold)
            if 'score' in data.columns:
                mp = pd.DataFrame({'s': data['score'].values[tr], 'y': y[tr]}).groupby('s')['y'].agg(lambda s: s.value_counts().idxmax()).to_dict()
                store('baseline: star rating', rep, te, [mp.get(s, maj) for s in data['score'].values[te]])
            # panjang review: ambang dipilih di train fold (<=thr -> positive, else negative)
            nw = data['n_words'].values; best = max(range(1, 30), key=lambda t: np.mean(np.where(nw[tr] <= t, 'positive', 'negative') == y[tr]))
            base = np.where(nw[te] <= best, 'positive', 'negative'); store('baseline: length rule', rep, te, base)
            # leksikon kontras: mixed jika ada kata kontras dan review >= 6 kata, selain itu aturan panjang
            hc = data['has_contrast'].values[te]; ln = nw[te]
            store('baseline: contrast lexicon (+len>=6)', rep, te, np.where(hc & (ln >= 6), 'mixed', base))
            store('baseline: contrast lexicon (any)', rep, te, np.where(hc, 'mixed', base))
            hcc = data['has_contrast_core'].values[te]      # hanya tapi/tetapi/namun (+ tp/tpi via normalisasi)
            store('baseline: contrast core tapi/tetapi/namun (+len>=6)', rep, te, np.where(hcc & (ln >= 6), 'mixed', base))
run_baselines()

## 6. TF-IDF classifiers — nested CV, semua model × semua strategi imbalance

In [9]:
class ROSWrap(BaseEstimator, ClassifierMixin):
    """Random oversampling di dalam fit (hanya data train yang diduplikasi; validasi/test tidak tersentuh)."""
    def __init__(self, estimator=None, ratio=1.0, seed=42): self.estimator = estimator; self.ratio = ratio; self.seed = seed
    def fit(self, X, yy):
        X = np.asarray(X, dtype=object); yy = np.asarray(yy); rng = np.random.default_rng(self.seed)
        cls, cnt = np.unique(yy, return_counts=True); target = int(cnt.max() * self.ratio); idx = []
        for c, n in zip(cls, cnt):
            ci = np.where(yy == c)[0]
            if n < target: ci = np.concatenate([ci, rng.choice(ci, target - n, replace=True)])
            idx.append(ci)
        idx = np.concatenate(idx); self.estimator_ = clone(self.estimator).fit(X[idx], yy[idx]); self.classes_ = self.estimator_.classes_; return self
    def predict(self, X): return self.estimator_.predict(X)

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAVE_SMOTE = CFG['use_smote']
except Exception as e:
    print('imbalanced-learn tidak tersedia -> SMOTE dilewati:', e); HAVE_SMOTE = False

def make_tfidf(): return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
def make_model(name, strategy):
    cw = strategy == 'class_weight'
    if name == 'MultinomialNB':  clf, grid = MultinomialNB(fit_prior=not cw), {'alpha': [0.1, 0.5, 1.0]}
    elif name == 'LinearSVM':    clf, grid = LinearSVC(class_weight='balanced' if cw else None, random_state=SEED), {'C': [0.1, 0.5, 1, 5]}
    elif name == 'LogReg':       clf, grid = LogisticRegression(class_weight='balanced' if cw else None, max_iter=2000), {'C': [1, 10, 100]}
    elif name == 'RandomForest': clf, grid = RandomForestClassifier(300, class_weight='balanced_subsample' if cw else None, n_jobs=-1, random_state=SEED), {'min_samples_leaf': [1, 2]}
    if strategy == 'smote':
        est = ImbPipeline([('tfidf', make_tfidf()), ('smote', SMOTE(k_neighbors=5, random_state=SEED)), ('clf', clf)]); pre = 'clf__'
    else:
        est = Pipeline([('tfidf', make_tfidf()), ('clf', clf)]); pre = 'clf__'
    if strategy == 'oversample':
        est = ROSWrap(est, CFG['ros_ratio'], SEED); pre = 'estimator__clf__'
    return est, {pre + k: v for k, v in grid.items()}

CLASSIC = ['MultinomialNB', 'LinearSVM', 'LogReg', 'RandomForest']
STRATS = ['none', 'class_weight', 'oversample'] + (['smote'] if HAVE_SMOTE else [])
f1m = make_scorer(f1_score, average='macro', labels=LABELS, zero_division=0)

def fit_predict_classic(name, strategy, text, tr, te, tune=True):
    est, grid = make_model(name, strategy)
    if tune:
        inner = list(StratifiedGroupKFold(CFG['inner_splits'], shuffle=True, random_state=SEED).split(np.zeros(len(tr)), y[tr], groups[tr]))
        gs = GridSearchCV(est, grid, scoring=f1m, cv=inner, n_jobs=-1, refit=True).fit(text[tr], y[tr])
        return gs.predict(text[te]), gs.best_params_
    return est.fit(text[tr], y[tr]).predict(text[te]), {}

def run_classic(variant, keys=None, repeats=None):
    text = data[f'text_ml_{variant}'].values; t0 = time.time(); best_params = {}
    for rep in range(repeats if repeats is not None else N_REP):
        for fi, (tr, te) in enumerate(SPLITS[rep]):
            for name in CLASSIC:
                for strat in STRATS:
                    key = f'{name} | {strat}'
                    if keys and key not in keys: continue
                    pred, bp = fit_predict_classic(name, strat, text, tr, te); store(key, rep, te, pred); best_params[(key, rep, fi)] = bp
        print(f'  repeat {rep} selesai ({time.time() - t0:.0f}s)')
    return best_params
print('Menjalankan model klasik (varian stopword utama =', CFG['stopword_variant'], ')')
BEST_PARAMS = run_classic(CFG['stopword_variant'])

Menjalankan model klasik (varian stopword utama = keep_negation_contrast )
  repeat 0 selesai (429s)
  repeat 1 selesai (863s)
  repeat 2 selesai (1310s)


### 7a. Ablasi stopword (LinearSVM, class_weight): apakah membuang kata kontras merugikan?

In [10]:
abl = []
for variant in ['nltk_default', 'keep_negation_contrast', 'no_stopwords']:
    text = data[f'text_ml_{variant}'].values; rows = []
    for rep in range(N_REP):
        oof = np.empty(len(y), dtype=object)
        for tr, te in SPLITS[rep]:
            oof[te] = fit_predict_classic('LinearSVM', 'class_weight', text, tr, te, tune=False)[0]
        rows.append(prf(y, oof))
    r = pd.DataFrame(rows); abl.append(dict(variant=variant, macro_f1=f'{r.macro_f1.mean():.3f}' + (f'±{r.macro_f1.std():.3f}' if len(r) > 1 else ''),
                                            F_mixed=f'{r.F_mixed.mean():.3f}' + (f'±{r.F_mixed.std():.3f}' if len(r) > 1 else ''), R_mixed=f'{r.R_mixed.mean():.3f}', acc=f'{r.acc.mean():.3f}'))
abl = pd.DataFrame(abl); print(abl.to_string(index=False)); abl.to_csv(f"{CFG['out_dir']}/ablation_stopwords.csv", index=False)

               variant    macro_f1     F_mixed R_mixed   acc
          nltk_default 0.710±0.006 0.219±0.015   0.165 0.946
keep_negation_contrast 0.787±0.015 0.443±0.041   0.337 0.952
          no_stopwords 0.785±0.001 0.416±0.004   0.286 0.963


### 7b. Pemeriksaan leakage duplikat: split acak vs split per-grup (LinearSVM)

In [11]:
text = data[f'text_ml_{CFG["stopword_variant"]}'].values; lk = []
for rep in range(N_REP):
    for name, splitter, grp in [('random KFold (duplikat bocor)', StratifiedKFold(CFG['n_splits'], shuffle=True, random_state=SEED + rep), None),
                                ('grouped KFold (dipakai di seluruh notebook)', StratifiedGroupKFold(CFG['n_splits'], shuffle=True, random_state=SEED + rep), groups)]:
        oof = np.empty(len(y), dtype=object)
        for tr, te in splitter.split(np.zeros(len(y)), y, grp):
            oof[te] = make_model('LinearSVM', 'class_weight')[0].fit(text[tr], y[tr]).predict(text[te])
        lk.append(dict(split=name, **prf(y, oof)))
lk = pd.DataFrame(lk).groupby('split')[['acc', 'macro_f1', 'F_mixed']].mean().round(4); print(lk); lk.to_csv(f"{CFG['out_dir']}/leakage_check.csv")

                                                acc  macro_f1  F_mixed
split                                                                 
grouped KFold (dipakai di seluruh notebook)  0.9518    0.7868   0.4432
random KFold (duplikat bocor)                0.9550    0.7830   0.4248


## 8. Transformer (IndoBERT, IndoBERTweet) — validation set internal, tanpa vs dengan *weighted loss*
Input = `content_clean` (tanpa stemming/stopword removal). Checkpoint terbaik dipilih pada **validation set internal** (10% dari train fold, group-aware); test fold hanya dipakai untuk prediksi akhir. Hyperparameter tetap (tidak di-tuning).

In [12]:
BERT_KEYS = []; BERT_GROUPS = {}
if CFG['run_bert']:
    import torch, shutil, inspect
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, Trainer, TrainingArguments,
                              DataCollatorWithPadding, set_seed)
    # versi transformers baru: warmup_ratio dihapus -> warmup_steps desimal (0.1 = 10% langkah)
    _WU = dict(warmup_ratio=0.1) if 'warmup_ratio' in inspect.signature(TrainingArguments.__init__).parameters else dict(warmup_steps=0.1)
    print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))
    if not torch.cuda.is_available(): print('PERINGATAN: tanpa GPU transformer akan sangat lambat. Runtime -> Change runtime type -> T4 GPU.')
    L2I = {l: i for i, l in enumerate(LABELS)}; I2L = {i: l for l, i in L2I.items()}
    class TextDS(torch.utils.data.Dataset):
        def __init__(self, texts, labels): self.enc = tok(list(texts), truncation=True, max_length=128); self.labels = [L2I[l] for l in labels]
        def __len__(self): return len(self.labels)
        def __getitem__(self, i): d = {k: v[i] for k, v in self.enc.items()}; d['labels'] = self.labels[i]; return d
    class WTrainer(Trainer):
        def __init__(self, *a, class_weights=None, **k): super().__init__(*a, **k); self.cw = class_weights
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop('labels'); out = model(**inputs)
            w = self.cw.to(out.logits.device) if self.cw is not None else None
            loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w)
            return (loss, out) if return_outputs else loss
    def bert_metrics(p):
        pr = p.predictions.argmax(-1); lab = p.label_ids
        return {'macro_f1': f1_score(lab, pr, average='macro', labels=[0, 1, 2], zero_division=0), 'acc': accuracy_score(lab, pr)}
    def run_bert_fold(hf_name, tr, te, seed, weighted, tag):
        k = int(round(1 / CFG['bert_val_frac']))
        sub_tr, sub_va = next(StratifiedGroupKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(tr)), y[tr], groups[tr]))
        tr_i, va_i = tr[sub_tr], tr[sub_va]
        ds_tr, ds_va, ds_te = [TextDS(data['text_bert'].values[i], y[i]) for i in (tr_i, va_i, te)]
        cfg = AutoConfig.from_pretrained(hf_name)
        cfg.num_labels = 3; cfg.id2label = I2L; cfg.label2id = L2I          # checkpoint bawaan 5 label -> paksa 3 label
        model = AutoModelForSequenceClassification.from_pretrained(hf_name, config=cfg)
        assert model.classifier.out_features == 3, 'kepala klasifikasi harus 3 kelas'
        cnt = np.bincount([L2I[l] for l in y[tr_i]], minlength=3); cw = torch.tensor(len(tr_i) / (3 * np.maximum(cnt, 1)), dtype=torch.float) if weighted else None
        out_dir = f"{CFG['out_dir']}/_bert_tmp_{tag}"
        args = TrainingArguments(output_dir=out_dir, learning_rate=CFG['bert_lr'], per_device_train_batch_size=CFG['bert_bs'],
                                 per_device_eval_batch_size=64, num_train_epochs=CFG['bert_epochs'], weight_decay=0.01, **_WU,
                                 eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, load_best_model_at_end=True,
                                 metric_for_best_model='macro_f1', greater_is_better=True, seed=seed, report_to='none',
                                 fp16=torch.cuda.is_available(), logging_steps=100)
        set_seed(seed)
        trn = WTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va, data_collator=DataCollatorWithPadding(tok),
                       compute_metrics=bert_metrics, class_weights=cw)
        trn.train(); pred = trn.predict(ds_te).predictions.argmax(-1)
        del trn, model; torch.cuda.empty_cache(); shutil.rmtree(out_dir, ignore_errors=True)
        return np.array([I2L[i] for i in pred])
    t0 = time.time()
    for short_name, hf_name in CFG['bert_models'].items():
        tok = AutoTokenizer.from_pretrained(hf_name); BERT_GROUPS[short_name] = []
        for weighted in [False, True]:
            key = f"{short_name} | {'weighted_loss' if weighted else 'none'}"; BERT_KEYS.append(key); BERT_GROUPS[short_name].append(key)
            for rep in range(CFG['bert_repeats']):
                for fi, (tr, te) in enumerate(SPLITS[rep]):
                    store(key, rep, te, run_bert_fold(hf_name, tr, te, SEED + 100 * rep + fi, weighted, f'{short_name}_{int(weighted)}_{rep}_{fi}'))
                    print(f'  {key} repeat {rep} fold {fi} ({time.time() - t0:.0f}s)')
else:
    print('run_bert=False -> bagian transformer dilewati')

GPU: Tesla T4


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

... (167 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  IndoBERTweet | weighted_loss repeat 0 fold 4 (3067s)


## 9. Baseline LLM (zero-shot & few-shot) — set `CFG['run_llm']=True`
Few-shot memakai dua set contoh yang **disjoint**: baris yang menjadi contoh di set A dievaluasi dengan prompt set B (dan sebaliknya), sehingga tidak ada baris yang dievaluasi dengan dirinya sendiri di prompt. Laporkan model, tanggal run, temperature (0), dan prompt.

In [13]:
LLM_KEYS = []
if CFG['run_llm']:
    import anthropic, datetime
    from concurrent.futures import ThreadPoolExecutor
    try:
        from google.colab import userdata; api_key = userdata.get('ANTHROPIC_API_KEY')
    except Exception: api_key = os.environ.get('ANTHROPIC_API_KEY')
    client = anthropic.Anthropic(api_key=api_key)
    INSTR = ("Klasifikasikan ulasan aplikasi Mobile JKN ke tepat satu label: positive (hanya pujian/puas), negative (hanya keluhan), "
             "mixed (memuat penilaian positif DAN negatif sekaligus). Jawab hanya dengan satu kata: positive, negative, atau mixed.")
    rng = np.random.default_rng(SEED); kshot = CFG['llm_shots_per_class']
    def pick(exclude):
        ids = [int(i) for l in LABELS for i in rng.choice([j for j in np.where(y == l)[0] if j not in exclude], kshot, replace=False)]
        rng.shuffle(ids); return ids
    exA = pick(set()); exB = pick(set(exA)); setA = set(exA)
    def prompt_for(mode, i):
        txt = f"Ulasan: {data['content'].iloc[i]}\nLabel:"
        if mode == 'zero_shot': return INSTR + '\n\n' + txt
        ex = exB if i in setA else exA            # baris contoh set A dievaluasi dengan contoh set B (dan sebaliknya)
        return INSTR + '\n\nContoh:\n\n' + ''.join(f"Ulasan: {data['content'].iloc[j]}\nLabel: {y[j]}\n\n" for j in ex) + txt
    def ask(prompt):
        last = None
        for attempt in range(4):
            try:
                r = client.messages.create(model=CFG['llm_model'], max_tokens=8, temperature=0, messages=[{'role': 'user', 'content': prompt}])
                t = r.content[0].text.strip().lower(); return next((l for l in LABELS if t.startswith(l)), None), None
            except Exception as e: last = e; time.sleep(2 ** attempt)
        return None, last
    for mode in CFG['llm_modes']:
        with ThreadPoolExecutor(8) as pool: res = list(pool.map(lambda i: ask(prompt_for(mode, i)), range(len(y))))
        fails = [r for r in res if r[0] is None]
        if len(fails) > 0.05 * len(y): raise RuntimeError(f'{mode}: {len(fails)}/{len(y)} panggilan gagal / tak terparse. Contoh error: {fails[0][1]}')
        pred = np.array([r[0] or 'positive' for r in res], dtype=object)      # gagal/tak terparse -> positive (dilaporkan)
        key = f"LLM {mode} | {CFG['llm_model']}"; OOF[key] = {0: pred}; LLM_KEYS.append(key)
        print(f'{key}: gagal/tak terparse {len(fails)} dari {len(y)}', {k: round(v, 3) for k, v in prf(y, pred).items()})
        pd.DataFrame({'row_id': data['row_id'], 'y': y, 'pred': pred}).to_csv(f"{CFG['out_dir']}/llm_predictions_{mode}.csv", index=False)
    LLM_DATE = str(datetime.date.today())
else:
    print('run_llm=False -> dilewati (aktifkan sebelum submit: reviewer hampir pasti menanyakan baseline LLM)')

run_llm=False -> dilewati (aktifkan sebelum submit: reviewer hampir pasti menanyakan baseline LLM)


## 10. Hasil utama (OOF gabungan; mean ± std antar-repeat; Wilson CI untuk mixed)
Catatan: transformer dan LLM dijalankan pada `bert_repeats`/1 repeat; tidak ada std untuk keduanya bila hanya 1 repeat.

In [14]:
def summarize():
    rows = []
    for key, reps in OOF.items():
        ms = [prf(y, reps[r]) for r in sorted(reps)]; m = pd.DataFrame(ms); p0 = reps[sorted(reps)[0]]
        k = int(((y == 'mixed') & (p0 == 'mixed')).sum()); n = int((y == 'mixed').sum()); lo, hi = wilson(k, n)
        pk = int((p0 == 'mixed').sum()); plo, phi = wilson(k, pk) if pk else (float('nan'), float('nan'))
        sd = lambda c: f'{m[c].mean():.3f}' + (f'±{m[c].std():.3f}' if len(m) > 1 else '')
        rows.append(dict(model=key, repeats=len(m), acc=sd('acc'), macro_f1=sd('macro_f1'), weighted_f1=sd('weighted_f1'), F_mixed=sd('F_mixed'),
                         P_mixed=sd('P_mixed'), R_mixed=sd('R_mixed'), R_mixed_CI95=f'[{lo:.2f},{hi:.2f}] ({k}/{n})',
                         P_mixed_CI95=f'[{plo:.2f},{phi:.2f}] ({k}/{pk})', _key=m['macro_f1'].mean()))
    return pd.DataFrame(rows).sort_values('_key', ascending=False).drop(columns='_key')
RES = summarize(); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); print(RES.to_string(index=False))
RES.to_csv(f"{CFG['out_dir']}/results_main.csv", index=False)

                                              model  repeats         acc    macro_f1 weighted_f1     F_mixed     P_mixed     R_mixed        R_mixed_CI95         P_mixed_CI95
                       IndoBERTweet | weighted_loss        1       0.976       0.898       0.977       0.733       0.660       0.824 [0.73,0.89] (70/85) [0.57,0.74] (70/106)
                                IndoBERTweet | none        1       0.975       0.894       0.975       0.724       0.708       0.741 [0.64,0.82] (63/85)  [0.61,0.79] (63/89)
                           IndoBERT | weighted_loss        1       0.974       0.890       0.974       0.714       0.670       0.765 [0.66,0.84] (65/85)  [0.57,0.76] (65/97)
                                    IndoBERT | none        1       0.974       0.881       0.974       0.686       0.667       0.706 [0.60,0.79] (60/85)  [0.56,0.76] (60/90)
                                     LogReg | smote        3 0.951±0.000 0.817±0.005 0.951±0.000 0.533±0.015 0.556±0.017 0.514±0.0

## 11. Uji signifikansi (cluster-bootstrap per grup duplikat + McNemar exact, koreksi Holm)
Semua perbandingan memakai prediksi OOF repeat 0 (satu-satunya repeat yang dimiliki semua model). p-value mentah dan terkoreksi Holm sama-sama dilaporkan.

In [15]:
def macro(yt, yp): return prf(yt, yp)['macro_f1']
def fmix(yt, yp): return prf(yt, yp)['F_mixed']
_gidx = pd.Series(np.arange(len(y))).groupby(groups).apply(lambda s: s.values).values
def cluster_bootstrap(pa, pb, metric, B=2000, seed=0):
    rng = np.random.default_rng(seed); d = []
    for _ in range(B):
        ix = np.concatenate(_gidx[rng.integers(0, len(_gidx), len(_gidx))]); d.append(metric(y[ix], pa[ix]) - metric(y[ix], pb[ix]))
    d = np.array(d); return metric(y, pa) - metric(y, pb), np.percentile(d, [2.5, 97.5]), min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()))
def mcnemar_exact(ca, cb):
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return b, c, (1.0 if b + c == 0 else stats.binomtest(min(b, c), b + c, 0.5).pvalue)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o): run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj
def compare(ka, kb):
    pa, pb = OOF[ka][0], OOF[kb][0]; r = dict(A=ka, B=kb)
    for nm, mt in [('macroF1', macro), ('F1_mixed', fmix)]:
        d, ci, p = cluster_bootstrap(pa, pb, mt); r[f'{nm}_delta'] = round(d, 3); r[f'{nm}_CI95'] = f'[{ci[0]:+.3f},{ci[1]:+.3f}]'; r[f'{nm}_p'] = p
    mm = y == 'mixed'; b, c, p = mcnemar_exact(pa[mm] == y[mm], pb[mm] == y[mm]); r['McNemar_mixed(b/c)'] = f'{b}/{c}'; r['McNemar_mixed_p'] = p
    b, c, p = mcnemar_exact(pa == y, pb == y); r['McNemar_all(b/c)'] = f'{b}/{c}'; r['McNemar_all_p'] = p
    return r
classic_keys = [k for k in OOF if k.split(' | ')[0] in CLASSIC]
def best_of(keys): return max(keys, key=lambda k: np.mean([prf(y, OOF[k][r])['macro_f1'] for r in OOF[k]]))
BEST_CLASSIC = best_of(classic_keys); print('Klasik terbaik (macro-F1):', BEST_CLASSIC)
BASE_LEX = 'baseline: contrast lexicon (+len>=6)'
pairs = [(BEST_CLASSIC, BASE_LEX)]; BEST_BERT = BEST_LLM = None
if BERT_KEYS:
    BEST_BERT = best_of(BERT_KEYS); print('Transformer terbaik:', BEST_BERT)
    pairs = [(BEST_BERT, BEST_CLASSIC), (BEST_BERT, BASE_LEX)] + pairs
    for m, ks in BERT_GROUPS.items(): pairs.append((ks[1], ks[0]))                         # weighted loss vs none
    if len(BERT_GROUPS) > 1: pairs.append(tuple(best_of(ks) for ks in list(BERT_GROUPS.values())[:2]))   # model A vs model B
    same = BEST_CLASSIC.split(' | ')[0] + ' | ' + ('class_weight' if 'weighted' in BEST_BERT else 'none')   # strategi imbalance sama
    if same in OOF: pairs.append((BEST_BERT, same))
if LLM_KEYS:
    BEST_LLM = best_of(LLM_KEYS); pairs.append((BEST_LLM, BEST_CLASSIC))
    if BEST_BERT: pairs.append((BEST_BERT, BEST_LLM))
seen = set(); sig = []
for a, b in pairs:
    if a in OOF and b in OOF and a != b and (a, b) not in seen: seen.add((a, b)); sig.append(compare(a, b))
SIG = pd.DataFrame(sig)
if len(SIG):
    SIG['macroF1_p_holm'] = holm(SIG['macroF1_p']); SIG['F1_mixed_p_holm'] = holm(SIG['F1_mixed_p'])
    SIG['McNemar_mixed_p_holm'] = holm(SIG['McNemar_mixed_p']); SIG['McNemar_all_p_holm'] = holm(SIG['McNemar_all_p'])
SIG = SIG.round(4); print(SIG.T.to_string()); SIG.to_csv(f"{CFG['out_dir']}/significance.csv", index=False)

Klasik terbaik (macro-F1): LogReg | smote
Transformer terbaik: IndoBERTweet | weighted_loss
                                                 0                                     1                                     2                         3                             4                             5                             6
A                     IndoBERTweet | weighted_loss          IndoBERTweet | weighted_loss                        LogReg | smote  IndoBERT | weighted_loss  IndoBERTweet | weighted_loss      IndoBERT | weighted_loss  IndoBERTweet | weighted_loss
B                                   LogReg | smote  baseline: contrast lexicon (+len>=6)  baseline: contrast lexicon (+len>=6)           IndoBERT | none           IndoBERTweet | none  IndoBERTweet | weighted_loss         LogReg | class_weight
macroF1_delta                                0.086                                 0.344                                 0.259                     0.009                         0.

## 12. Analisis per panjang review & per tingkat slang

In [16]:
bucket = pd.cut(data['n_words'], [0, 3, 10, 10_000], labels=['≤3 kata', '4–10 kata', '>10 kata']).astype(str)
show = [k for k in [BEST_CLASSIC, *BERT_KEYS, *LLM_KEYS, 'baseline: majority', 'baseline: length rule', BASE_LEX] if k in OOF]
rows = []
for k in show:
    p = OOF[k][0]
    for b in ['≤3 kata', '4–10 kata', '>10 kata']:
        m = (bucket == b).values; mm = m & (y == 'mixed')
        rows.append(dict(model=k, bucket=b, n=int(m.sum()), n_mixed=int(mm.sum()), acc=accuracy_score(y[m], p[m]),
                         macro_f1=f1_score(y[m], p[m], average='macro', labels=[l for l in LABELS if (y[m] == l).any()], zero_division=0),
                         R_mixed=(p[mm] == 'mixed').mean() if mm.sum() else np.nan))
LEN = pd.DataFrame(rows).round(3); print(LEN.to_string(index=False)); LEN.to_csv(f"{CFG['out_dir']}/by_length.csv", index=False)
if 'slang_density_tier' in data.columns:
    rows = []
    for k in show[:2]:
        for s, g in data.groupby('slang_density_tier'):
            ix = g.index.values; rows.append(dict(model=k, slang=s, n=len(ix), acc=accuracy_score(y[ix], OOF[k][0][ix])))
    print(pd.DataFrame(rows).round(3).to_string(index=False))

                               model    bucket    n  n_mixed   acc  macro_f1  R_mixed
                      LogReg | smote   ≤3 kata 1773        0 0.957     0.910      NaN
                      LogReg | smote 4–10 kata 1448       19 0.954     0.849    0.632
                      LogReg | smote  >10 kata 1553       66 0.943     0.776    0.439
                     IndoBERT | none   ≤3 kata 1773        0 0.987     0.974      NaN
                     IndoBERT | none 4–10 kata 1448       19 0.970     0.890    0.684
                     IndoBERT | none  >10 kata 1553       66 0.961     0.863    0.712
            IndoBERT | weighted_loss   ≤3 kata 1773        0 0.986     0.972      NaN
            IndoBERT | weighted_loss 4–10 kata 1448       19 0.971     0.908    0.737
            IndoBERT | weighted_loss  >10 kata 1553       66 0.962     0.869    0.773
                 IndoBERTweet | none   ≤3 kata 1773        0 0.990     0.980      NaN
                 IndoBERTweet | none 4–10 kata 1448   

## 13. Error analysis kelas mixed (pola + contoh kualitatif)
Tabel pola menunjukkan *apa yang membedakan* mixed yang terdeteksi vs terlewat. CSV `error_analysis_mixed.csv` memuat semua kesalahan dengan kolom kosong `pola_manual`: kodekan secara manual (usulan kategori: *implisit tanpa kata kontras*, *sarkasme*, *pujian + keluhan sangat pendek*, *urutan pujian→keluhan dalam kalimat panjang*, *label ambigu / kandidat noise anotasi*) lalu laporkan frekuensinya + 3–5 contoh di paper.

In [17]:
err_rows = []
for key in [k for k in [BEST_CLASSIC, BEST_BERT, BEST_LLM] if k]:
    p = OOF[key][0]; d = data.assign(pred=p); mm = d[d.label_annotator1 == 'mixed']
    hit, miss = mm[mm.pred == 'mixed'], mm[mm.pred != 'mixed']; fp = d[(d.label_annotator1 != 'mixed') & (d.pred == 'mixed')]
    def prof(g): return dict(n=len(g), kontras_any_pct=round(100 * g.has_contrast.mean(), 0) if len(g) else np.nan, kontras_inti_pct=round(100 * g.has_contrast_core.mean(), 0) if len(g) else np.nan,
                             median_kata=g.n_words.median() if len(g) else np.nan, **({'rating_mean': round(g.score.mean(), 2)} if 'score' in g else {}))
    print(f'\n=== {key} ===')
    print(pd.DataFrame({'mixed terdeteksi': prof(hit), 'mixed terlewat': prof(miss), 'salah-prediksi sbg mixed': prof(fp)}).T.to_string())
    print('Mixed terlewat -> diprediksi sebagai:', miss.pred.value_counts().to_dict(), '| false positive berasal dari:', fp.label_annotator1.value_counts().to_dict())
    print(miss[['content', 'pred']].sample(min(8, len(miss)), random_state=SEED).to_string(index=False))
    cols = ['row_id', 'content', 'content_clean', 'label_annotator1', 'pred', 'n_words', 'has_contrast', 'has_contrast_core'] + (['score'] if 'score' in data.columns else [])
    err_rows.append(pd.concat([miss.assign(error='false_negative_mixed'), fp.assign(error='false_positive_mixed')])[cols + ['error']].assign(model=key, pola_manual=''))
pd.concat(err_rows).to_csv(f"{CFG['out_dir']}/error_analysis_mixed.csv", index=False)


=== LogReg | smote ===
                             n  kontras_any_pct  kontras_inti_pct  median_kata  rating_mean
mixed terdeteksi          41.0            100.0              80.0         17.0         3.34
mixed terlewat            44.0             77.0              50.0         19.5         3.52
salah-prediksi sbg mixed  31.0             90.0              68.0         20.0         2.10
Mixed terlewat -> diprediksi sebagai: {'negative': 34, 'positive': 10} | false positive berasal dari: {'negative': 25, 'positive': 6}
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

## 14. Gambar untuk paper

In [18]:
fig, axs = plt.subplots(1, len(show[:3]), figsize=(4.2 * len(show[:3]), 3.8), dpi=200)
axs = np.atleast_1d(axs)
for ax, k in zip(axs, show[:3]):
    cm = confusion_matrix(y, OOF[k][0], labels=LABELS); rn = cm / cm.sum(1, keepdims=True)
    ax.imshow(rn, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3): ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', color='white' if rn[i, j] > .55 else 'black')
    ax.set_xticks(range(3)); ax.set_xticklabels(['pos', 'neg', 'mix']); ax.set_yticks(range(3)); ax.set_yticklabels(['pos', 'neg', 'mix'])
    ax.set_title(k, fontsize=8); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/fig_confusion.png"); plt.close()

keys = [k for k in RES.model if not k.startswith('baseline')][:10]; vals = []
for k in keys:
    kk = int(((y == 'mixed') & (OOF[k][0] == 'mixed')).sum()); n = int((y == 'mixed').sum()); lo, hi = wilson(kk, n); vals.append((kk / n, lo, hi))
fig, ax = plt.subplots(figsize=(7, 0.45 * len(keys) + 1), dpi=200)
for i, (k, (v, lo, hi)) in enumerate(zip(keys[::-1], vals[::-1])):
    ax.plot([lo, hi], [i, i], color='#9ca3af', lw=2); ax.plot(v, i, 'o', color='#2563eb', mec='white')
ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys[::-1], fontsize=8); ax.set_xlabel('Recall kelas mixed (OOF, Wilson 95% CI)'); ax.grid(axis='x', color='#e5e7eb')
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/fig_mixed_recall.png"); plt.close()

## 15. Reproducibility & fakta untuk paper
Sel ini menulis `paper_facts.md`: angka eksak untuk bagian Data, Annotation, dan Limitations.

In [20]:
import platform, sklearn
info = dict(python=platform.python_version(), sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__, seed=SEED, config=CFG,
            n_rows=len(data), n_groups=int(len(set(groups))), class_counts=pd.Series(y).value_counts().to_dict(), sastrawi=STEM_OK)
try:
    import transformers, torch; info.update(transformers=transformers.__version__, torch=torch.__version__)
except Exception: pass
if CFG['run_llm']: info.update(llm_model=CFG['llm_model'], llm_run_date=LLM_DATE, llm_temperature=0)
json.dump(info, open(f"{CFG['out_dir']}/run_info.json", 'w'), indent=1, default=str)

F = ['# Fakta untuk paper (dihasilkan otomatis)\n']
F.append('## Data & filter\n```\n' + FUNNEL.to_string(index=False) + '\n```')
F.append(f"- Data final: {len(data)} review; kelas: {pd.Series(y).value_counts().to_dict()}; {len(data) - len(set(groups))} baris duplikat teks (dikelompokkan agar tidak terpisah train/test).")
F.append("- Filter sebelum anotasi hanya membuang review ≤3 karakter setelah cleaning" + (" dan label otomatis neutral (berisiko membuang sebagian mixed; tidak dapat dicek)." if CFG['drop_auto_neutral'] else ". Review berlabel otomatis neutral tetap dianotasi, jadi tidak ada seleksi berbasis label otomatis."))
F.append("- `neutral` dikeluarkan karena tugas dibatasi pada tiga kelas; ini membuat tugas lebih sederhana daripada kondisi nyata.")
if DATE_COL:
    dts = pd.to_datetime(raw[DATE_COL], errors='coerce')
    F.append(f"- Rentang waktu: {dts.min():%d %B %Y} – {dts.max():%d %B %Y} ({(dts.max() - dts.min()).days + 1} hari), satu aplikasi (Mobile JKN), satu platform (isi nama platform sumber data). Generalisasi ke periode/aplikasi lain tidak diuji.")
F.append('\n## Anotasi')
F.append("- Protokol: annotator hanya melihat teks ulasan (`content`, `content_clean`); rating, label otomatis Kaggle, dan rationale TIDAK ditampilkan (blind). Verifikasi: file anotasi tidak memuat kolom tersebut.")
if kap:
    km = kap['kappa_mixed_vs_rest']
    F.append(f"- Subset ganda: n={kap['n_double_annotated']} ({kap['pct_of_all_annotated']}% dari yang dianotasi); kategori: {kap['categories']}; kappa semua kategori = {kap['kappa_all_categories']:.3f} (CI95 {kap['kappa_all_categories_CI95'][0]:.3f}–{kap['kappa_all_categories_CI95'][1]:.3f}).")
    F.append(f"- Kappa kelas mixed (one-vs-rest) = {km['kappa']:.3f} (CI95 {km['CI95'][0]:.3f}–{km['CI95'][1]:.3f}); mixed menurut A1 = {km['n_A1']}, A2 = {km['n_A2']}, keduanya = {km['n_both']}. Gunakan angka ini, bukan kappa keseluruhan, untuk klaim reliabilitas kelas mixed.")
open(f"{CFG['out_dir']}/paper_facts.md", 'w').write('\n'.join(F))
print('\n'.join(F))
print('\nFile hasil ada di folder', CFG['out_dir'], '- unduh: results_main.csv, significance.csv, by_length.csv, ablation_stopwords.csv, error_analysis_mixed.csv, kappa.json, data_funnel.csv, paper_facts.md, fig_*.png')

# Fakta untuk paper (dihasilkan otomatis)

## Data & filter
```
                                        langkah    n
                         Review mentah (Kaggle) 5000
− terlalu pendek setelah cleaning (≤3 karakter) -198
                                = masuk anotasi 4802
       (di antaranya berlabel otomatis neutral)   92
                        Baris pada file anotasi 4802
                        − Annotator 1 = neutral  -28
     = data final (positive / negative / mixed) 4774
```
- Data final: 4774 review; kelas: {'positive': 2387, 'negative': 2302, 'mixed': 85}; 1046 baris duplikat teks (dikelompokkan agar tidak terpisah train/test).
- Filter sebelum anotasi hanya membuang review ≤3 karakter setelah cleaning. Review berlabel otomatis neutral tetap dianotasi, jadi tidak ada seleksi berbasis label otomatis.
- `neutral` dikeluarkan karena tugas dibatasi pada tiga kelas; ini membuat tugas lebih sederhana daripada kondisi nyata.
- Rentang waktu: 09 September 2026 – 17 October 2026 (